# Low-Side PySyft Client — Submit Code to High-Side Datasite

This notebook accompanies the IEEE IRI 2026 paper *"A Privacy-Preserving Framework
Using Remote Data Science for Inter-Institutional Student Retention Prediction"*
([repo](https://github.com/jtfields/NAIRR240195-Privacy-Preserving-Machine-Learning)).

## Purpose

Demonstrates the **researcher-side workflow** in the Remote Data Science (RDS)
framework: connect to the low-side PySyft datasite, browse mock data, develop and
test a classification function locally on mock data, then submit it as a code
request for the data owner to review and execute on the high-side private data.

This is the client half of the RDS workflow described in Section III.D of the
paper.

## ⚠️ Prerequisites

**This notebook requires a live PySyft datasite to connect to.** It cannot be run
standalone from the repository. You will need:

- A running PySyft server (low-side) with the "Faketucky Mock Dataset" registered
- Login credentials provided by the data owner
- PySyft 0.9.2 (matches the paper's deployment)

Without these, the notebook serves as documentation of the PySyft client pattern.

## Workflow

1. Install and import PySyft
2. Login to the low-side datasite
3. Browse available datasets and load mock features/targets
4. Define a classification function with all dependencies imported inside
5. Test locally on mock data to catch errors before submission
6. Wrap the function with `syft_function_single_use` for remote execution
7. Create a project and submit the code as a request for data owner review
8. Check request status

Once approved by the data owner, the function executes on the high-side private
data and only aggregate metrics (or DP-protected row-level outputs) are returned.

## Note on the model function

The `faketucky_model()` function (cell 12) imports all its dependencies inside the
function body. This is required for PySyft remote execution — the function ships
its dependencies with it. The SMOTE step is commented out because that package
was not available on the high-side at the time of the paper's experiments.

## 1. Install and Import PySyft

In [ ]:
!pip install syft

In [ ]:
import syft as sy
import pandas as pd

## 2. Login to Low-Side Datasite

In [ ]:
import os
import getpass

# PySyft server credentials must be supplied by the data owner.
# Set these as environment variables before running, OR enter them when prompted.
SERVER_URL = os.environ.get("PYSYFT_URL") or input("PySyft server URL: ")
EMAIL = os.environ.get("PYSYFT_EMAIL") or input("Email: ")
PASSWORD = os.environ.get("PYSYFT_PASSWORD") or getpass.getpass("Password: ")

my_client = sy.login(url=SERVER_URL, email=EMAIL, password=PASSWORD)

## 3. Browse Available Datasets

In [ ]:
my_client.datasets

In [ ]:
ft_dataset = my_client.datasets["Faketucky Mock Dataset"]

In [ ]:
ft_dataset

## 4. Load Mock Features and Targets

In [ ]:
features, targets = ft_dataset.assets  # using Python tuple unpacking

In [ ]:
features.mock.head()  # pandas.DataFrame

In [ ]:
targets.mock.head()

In [ ]:
features.data

In [ ]:
targets.data

In [ ]:
X = features.mock
y = targets.mock

## 5. Define Classification Function for Remote Execution

In [ ]:
def faketucky_model(features, target):
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    # import seaborn as sns
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, OrdinalEncoder
    from sklearn.impute import SimpleImputer
    # from imblearn.over_sampling import SMOTE
    from xgboost import XGBClassifier
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    from sklearn.metrics import classification_report, confusion_matrix,roc_curve, auc, precision_recall_curve
    import warnings
    warnings.filterwarnings('ignore')
    SEED = 42
    np.random.seed(SEED)

    # Check if target and features are valid
    if target is None or target.empty:
        print("Error: Target Series is None or empty!")
        return

    if features is None or features.empty:
        print("Error: Features DataFrame is None or empty!")
        return

    print("Target variable distribution:")
    print(target.value_counts())
    print(f"Retention rate: {target.mean() * 100:.2f}%")

    print("Features overview:")
    print(features.head())

    print("Data types of features:")
    print(features.dtypes.value_counts())

    # Make a copy of the features to avoid modifying the original
    features_processed = features.copy()

    # Find any columns that are string-like but not labeled as 'object' type
    additional_categorical_cols = []
    for col in features_processed.columns:
        # Check sample values to identify string columns regardless of their dtype
        if features_processed[col].notna().any():  # Only check columns with non-NA values
            sample_val = features_processed[col].dropna().iloc[0]
            if isinstance(sample_val, str):
                additional_categorical_cols.append(col)

    # Combine known categorical columns with additional ones found
    categorical_cols = features_processed.select_dtypes(include=['object', 'category']).columns.tolist()
    # Add any string columns that weren't caught by the dtype check
    for col in additional_categorical_cols:
        if col not in categorical_cols:
            categorical_cols.append(col)

    print(f"Categorical columns identified: {categorical_cols}")

    # Convert all categorical columns to object type to ensure proper encoding
    for col in categorical_cols:
        features_processed[col] = features_processed[col].astype('object')

    # Handle categorical columns
    if categorical_cols:
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        features_processed[categorical_cols] = encoder.fit_transform(features_processed[categorical_cols])
        print("Features after encoding:")
        print(features_processed.head())

    # Handle missing values
    imputer = SimpleImputer(strategy='most_frequent')
    features_imputed = imputer.fit_transform(features_processed)

    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features_imputed)

    # Split into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(features_scaled, target, test_size=0.2, random_state=SEED)
    print(f"Split into train and test sets: {X_train.shape[0]} training samples, {X_test.shape[0]} test samples.")

    # Train XGBoost model
    model = XGBClassifier(random_state=SEED)
    model.fit(X_train, y_train)
    print("XGBoost model trained successfully.")

    # Evaluate model
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print("Model Evaluation Metrics:")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1 Score: {f1:.2f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.show()

    # Precision-Recall Curve
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, model.predict_proba(X_test)[:, 1])
    plt.figure(figsize=(8, 6))
    plt.plot(recall_vals, precision_vals, color='blue', label='Precision-Recall Curve')
    plt.title('Precision-Recall Curve')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend(loc='lower left')
    plt.show()

    # Find optimal threshold
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    thresholds = np.linspace(0, 1, 100)
    f1_scores = [f1_score(y_test, (y_pred_proba >= t).astype(int)) for t in thresholds]

    optimal_threshold = thresholds[np.argmax(f1_scores)]
    print(f"Optimal threshold for F1 score: {optimal_threshold:.4f}")
    print(f"F1 score at optimal threshold: {max(f1_scores):.4f}")

    y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)
    print("Classification Report with Optimal Threshold:")
    print(classification_report(y_test, y_pred_optimal))

    # Plot performance metrics vs. threshold
    plt.figure(figsize=(12, 6))
    plt.plot(thresholds, f1_scores, label='F1 Score')
    plt.xlabel('Threshold')
    plt.ylabel('Score')
    plt.title('Performance Metrics vs. Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # Confusion matrix and classification report
    if y_pred is not None:
      print("Confusion Matrix:")
      print(confusion_matrix(y_test, y_pred))
      print("\nClassification Report:")
      print(classification_report(y_test, y_pred))
    else:
      print("Error: Predictions are not available.")

    return accuracy,precision,recall,f1

## 6. Test Locally on Mock Data

In [ ]:
faketucky_model(features=features.mock, target=targets.mock)

## 7. Wrap Function for Remote Execution

In [ ]:
remote_user_code = sy.syft_function_single_use(features_data=features, labels=targets)(faketucky_model)

## 8. Create Project and Submit Code Request

In [ ]:
description = """
    The purpose of this study will be to run a machine learning
    experimental pipeline on faketucky data.
"""

# Create a project

new_project = my_client.create_project(
    name="Faketucky data",
    description=description,
    user_email_address=EMAIL  # reuse from cell 2
)

## 9. Check Status

In [ ]:
my_client.projects

In [ ]:
code_request = new_project.create_code_request(remote_user_code, my_client)

In [ ]:
code_request

In [ ]:
my_client.code

In [ ]:
my_client.requests